# 30 metrics (g5)

How long it takes. What you should see: every join and leave between the two newest EirGrid editions confirmed by the PDFs' own text; then pick 10 and check them by hand in the two PDFs. Then how much of each batch's distribution MW is energised, how long transmission projects have waited, and the exhibit.

In [ ]:
import subprocess

from iegrid import metrics, store
from iegrid.checks import transitions_table

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: transitions between the two newest editions, confirmed by the PDFs, 10 for a hand check

In [ ]:
rows = transitions_table(con)
for r in rows:
    print(f"{r[0]:15} {r[1]:26} {r[2]:7} {r[3]:7} {r[4]:40} {r[5]:7.1f} MW  confirmed {r[6]}")
unconfirmed = [r for r in rows if not r[6]]
assert rows, "no transitions"
assert not unconfirmed, f"transitions the PDFs don't confirm: {unconfirmed}"
print(f"\n{len(rows)} transitions, every one confirmed by the PDF text")
print("hand check, in the two PDFs:")
for r in rows[:: max(1, len(rows) // 10)][:10]:
    print(f"  {r[2]:7} {r[3]:7} {r[4]}")

## batch results dates (the earliest results publication stored)

In [ ]:
for b, d in metrics.batch_dates(con).items():
    print(f'{b:8} {d:%b %Y}')

## distribution burn-down by batch

In [ ]:
for b in metrics.dso_burndown(con):
    print(f"{b.batch:8} {b.result:%b %Y}  energised {float(b.energised_mw):7.1f} MW ({b.energised_n:3})"
          f"  contracted {float(b.contracted_mw):7.1f} MW ({b.contracted_n:3})  {b.share:5.1%}"
          f"  median months to energise {b.median_months_to_energise}")

## transmission: how long projects still contracted have waited (from their earliest signed month)

In [ ]:
for w in metrics.tso_waiting(con):
    print(f"{w.batch:26} {w.projects:3} projects {float(w.mw):7.0f} MW  median {w.median_months:5.1f} months  longest {w.longest_months}")

## exhibit: how long it takes

In [ ]:
from iegrid import exhibits

fig = exhibits.batch_burndown(con)
exhibits.save(fig, "g5_batch_burndown")